# Baseline notebook: customer churn
Non-ML rule baseline vs a simple logistic regression, evaluated with explicit false-positive / false-negative costs. **Only 12 rows: every number here is illustrative, not evidence of deployable performance.**

In [ ]:
import pandas as pd, numpy as np
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, brier_score_loss, roc_auc_score
df=pd.read_csv('customer-churn-training.csv')
print(df.shape, df.isna().sum().sum(), 'missing;', df.duplicated().sum(), 'duplicate rows')
print(df.churned.value_counts())

(12, 7) 0 missing; 0 duplicate rows
churned
0    7
1    5
Name: count, dtype: int64


## Cost assumptions (to be replaced with finance numbers)
- False negative (missed churner, no retention offer): lose ~12 months of spend. Using avg churner spend ≈ ₹619/month → ≈ ₹7,400.
- False positive (offer sent to a customer who would have stayed): discount/outreach cost ≈ ₹300.
- Ratio FN:FP ≈ 25:1, so the decision threshold should favour recall.

In [ ]:
FN_COST, FP_COST = 7400, 300
def cost(y, p):
    tn,fp,fn,tp = confusion_matrix(y,p,labels=[0,1]).ravel()
    return fn*FN_COST + fp*FP_COST, dict(tn=tn,fp=fp,fn=fn,tp=tp)
y=df.churned.values

## Non-ML baselines
1. Do nothing (predict nobody churns). 2. Rule: flag if no login in >=7 days OR >=3 support tickets.

In [ ]:
print('do nothing  ', cost(y, np.zeros(len(y),int)))
rule=((df.last_login_days>=7)|(df.support_tickets>=3)).astype(int).values
print('rule baseline', cost(y, rule))
print(df.assign(rule=rule)[['customer_id','last_login_days','support_tickets','churned','rule']])

do nothing   (np.int64(37000), {'tn': np.int64(7), 'fp': np.int64(0), 'fn': np.int64(5), 'tp': np.int64(0)})
rule baseline (np.int64(300), {'tn': np.int64(6), 'fp': np.int64(1), 'fn': np.int64(0), 'tp': np.int64(5)})
   customer_id  last_login_days  support_tickets  churned  rule
0         C001               21                4        1     1
1         C002                2                1        0     0
2         C003               14                3        1     1
3         C004                1                0        0     0
4         C005                5                2        0     0
5         C006               30                5        1     1
6         C007                3                1        0     0
7         C008               17                4        1     1
8         C009                4                2        0     0
9         C010               12                3        1     1
10        C011                2                0        0     0
11        C012 

## Simple model, leave-one-out CV
Numeric behavioural features only. `plan_type` is excluded on purpose: in this sample it separates the classes perfectly (every Basic customer churned), which a 12-row sample cannot distinguish from coincidence, and it is a price-tier proxy for `monthly_spend_inr`.

In [ ]:
X=df[['tenure_months','support_tickets','last_login_days']]
m=make_pipeline(StandardScaler(), LogisticRegression(C=0.5))
proba=cross_val_predict(m,X,y,cv=LeaveOneOut(),method='predict_proba')[:,1]
for t in [0.5,0.3,0.15]:
    print('threshold',t, cost(y,(proba>=t).astype(int)))
print('LOO AUC', round(roc_auc_score(y,proba),3), ' Brier', round(brier_score_loss(y,proba),3))
print(pd.DataFrame({'id':df.customer_id,'y':y,'p':proba.round(2)}))

threshold 0.5 (np.int64(0), {'tn': np.int64(7), 'fp': np.int64(0), 'fn': np.int64(0), 'tp': np.int64(5)})
threshold 0.3 (np.int64(600), {'tn': np.int64(5), 'fp': np.int64(2), 'fn': np.int64(0), 'tp': np.int64(5)})
threshold 0.15 (np.int64(900), {'tn': np.int64(4), 'fp': np.int64(3), 'fn': np.int64(0), 'tp': np.int64(5)})
LOO AUC 1.0  Brier 0.082
      id  y     p
0   C001  1  0.83
1   C002  0  0.13
2   C003  1  0.55
3   C004  0  0.03
4   C005  0  0.33
5   C006  1  0.97
6   C007  0  0.10
7   C008  1  0.69
8   C009  0  0.26
9   C010  1  0.54
10  C011  0  0.07
11  C012  0  0.49


## Takeaways
- Both the rule and the model are compared against doing nothing using cost, not accuracy.
- With 12 rows, AUC and cost estimates have huge uncertainty; the real next step is collecting more data, not tuning.
- The rule baseline is simple, auditable and may be hard to beat; ML is justified only if it beats it on a larger time-split holdout.